<a href="https://colab.research.google.com/github/ishikabeniwal/CSET343/blob/main/Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from scipy import signal
from scipy.stats import ttest_ind, chi2_contingency, f_oneway

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully")

In [ ]:
# Change this path if your CSV has a different name/location
TABULAR_PATH = "diabetes.csv"

df = pd.read_csv(TABULAR_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Zero counts before cleaning:")
for col in zero_as_missing:
    print(col, ":", (df[col] == 0).sum())

df_clean = df.copy()

for col in zero_as_missing:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print("\nMissing values after converting implausible zeros:")
print(df_clean.isnull().sum())

In [ ]:
# Fill missing numeric values using the median of each column
numeric_cols = df_clean.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Missing values after median imputation:")
print(df_clean.isnull().sum())

In [ ]:
print(df_clean.describe())

df_clean["Outcome"].value_counts().plot(kind="bar")
plt.title("Diabetes Outcome Class Distribution")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.show()

In [ ]:
df_clean.hist(figsize=(12, 8))
plt.suptitle("Pima Dataset Feature Distributions")
plt.tight_layout()
plt.show()

In [ ]:
def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

df_clean["BMI_Category"] = df_clean["BMI"].apply(bmi_category)
print(df_clean[["BMI", "BMI_Category"]].head())

In [ ]:
X = df_clean.drop(columns=["Outcome", "BMI_Category"])
y = df_clean["Outcome"]

selector = SelectKBest(score_func=f_classif, k=min(5, X.shape[1]))
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected features:", list(selected_features))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

print("Scaled data shape:", X_scaled.shape)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Feature Extraction")
plt.show()

### A6. Hypothesis Test

**Example:** Compare mean glucose levels between the two outcome groups.

- H0: The mean glucose level is the same in both groups.
- H1: The mean glucose level is different between the groups.

In [ ]:
group0 = df_clean[df_clean["Outcome"] == 0]["Glucose"]
group1 = df_clean[df_clean["Outcome"] == 1]["Glucose"]

t_stat, p_value = ttest_ind(group0, group1, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Reject H0: there is a statistically significant difference.")
else:
    print("Fail to reject H0: sufficient evidence of a difference was not found.")

### A7. Chi-square Test

Test whether **BMI category** and **Outcome** are associated.

- H0: BMI category and diabetes outcome are independent.
- H1: BMI category and diabetes outcome are associated.

In [ ]:
contingency_table = pd.crosstab(df_clean["BMI_Category"], df_clean["Outcome"])
display(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("Reject H0: the variables show a statistically significant association.")
else:
    print("Fail to reject H0: sufficient evidence of association was not found.")

### A8. ANOVA

Compare mean glucose levels across BMI categories.

- H0: All group means are equal.
- H1: At least one group mean is different.

In [ ]:
anova_groups = []

for category in df_clean["BMI_Category"].unique():
    values = df_clean[df_clean["BMI_Category"] == category]["Glucose"].values
    if len(values) > 0:
        anova_groups.append(values)

f_stat, p_value = f_oneway(*anova_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject H0: at least one group mean is significantly different.")
else:
    print("Fail to reject H0: sufficient evidence of a difference was not found.")

In [ ]:
TEXT_PATH = "mtsamples.csv"

text_df = pd.read_csv(TEXT_PATH)
print("Shape:", text_df.shape)
display(text_df.head())

possible_text_columns = [
    "transcription", "Transcription", "text", "Text",
    "medical_transcription", "description", "Description"
]

text_col = next((c for c in possible_text_columns if c in text_df.columns), None)

if text_col is None:
    print("Text column was not automatically identified.")
    print("Available columns:", list(text_df.columns))
else:
    print("Using text column:", text_col)

In [ ]:
def clean_medical_text(text):
    text = str(text)

    # Remove common direct identifiers
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', ' ', text)
    text = re.sub(r'\b(?:\+?\d[\d\s().-]{7,}\d)\b', ' ', text)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', ' ', text)

    # Normalize whitespace and lowercase
    text = re.sub(r'\s+', ' ', text).strip().lower()

    return text

if text_col is not None:
    text_df["clean_text"] = text_df[text_col].apply(clean_medical_text)
    display(text_df[[text_col, "clean_text"]].head())

In [ ]:
if text_col is not None:
    text_df["word_count"] = text_df["clean_text"].str.split().str.len()

    text_df["word_count"].hist(bins=30)
    plt.title("Medical Transcription Word Count")
    plt.xlabel("Number of Words")
    plt.ylabel("Frequency")
    plt.show()

    print(text_df["word_count"].describe())

In [ ]:
if text_col is not None:
    sample_text = text_df["clean_text"].iloc[0]
    tokens = sample_text.split()

    print("First 30 tokens:")
    print(tokens[:30])
    print("Number of tokens:", len(tokens))

In [ ]:
IMAGE_DIR = "chest_xray"

# Change these directories if your dataset uses a different structure
possible_dirs = [
    os.path.join(IMAGE_DIR, "train"),
    os.path.join(IMAGE_DIR, "test"),
    os.path.join(IMAGE_DIR, "val")
]

for d in possible_dirs:
    print(d, "exists:", os.path.exists(d))

In [ ]:
# Find a few image files recursively
image_files = []

if os.path.exists(IMAGE_DIR):
    for root, dirs, files in os.walk(IMAGE_DIR):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                image_files.append(os.path.join(root, file))

print("Images found:", len(image_files))
print("First few:", image_files[:5])

In [ ]:
if image_files:
    img = Image.open(image_files[0]).convert("L")

    plt.imshow(img, cmap="gray")
    plt.title("Sample Chest X-Ray")
    plt.axis("off")
    plt.show()

    print("Original image size:", img.size)
    print("Image mode:", img.mode)

In [ ]:
TARGET_SIZE = (224, 224)

def preprocess_xray(path):
    img = Image.open(path).convert("L")
    img = img.resize(TARGET_SIZE)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = np.expand_dims(arr, axis=-1)
    return arr

if image_files:
    xray_array = preprocess_xray(image_files[0])
    print("Processed shape:", xray_array.shape)
    print("Minimum:", xray_array.min())
    print("Maximum:", xray_array.max())

In [ ]:
if image_files:
    original = Image.open(image_files[0]).convert("L").resize(TARGET_SIZE)
    augmented = original.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

    fig = plt.figure(figsize=(8, 4))

    ax1 = fig.add_axes([0, 0, 0.45, 1])
    ax1.imshow(original, cmap="gray")
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = fig.add_axes([0.55, 0, 0.45, 1])
    ax2.imshow(augmented, cmap="gray")
    ax2.set_title("Augmented: Horizontal Flip")
    ax2.axis("off")

    plt.show()

In [ ]:
print("Privacy checklist for medical images:")
print("1. Remove patient name/ID and other direct identifiers from DICOM metadata.")
print("2. Check for burned-in identifiers inside the image itself.")
print("3. Keep only metadata required for the analysis.")
print("4. Follow the dataset's licensing and privacy requirements.")

In [ ]:
# If using the WFDB package:
# pip install wfdb

try:
    import wfdb
    print("WFDB imported successfully")
except ImportError:
    print("WFDB is not installed. Run: pip install wfdb")

In [ ]:
RECORD_PATH = "100"  # Change to your local record path, e.g. "mitdb/100"

if "wfdb" in globals():
    try:
        record = wfdb.rdrecord(RECORD_PATH)
        print("Signal shape:", record.p_signal.shape)
        print("Sampling frequency:", record.fs)
        print("Number of channels:", record.n_sig)
    except Exception as e:
        print("Record not loaded. Set RECORD_PATH to your downloaded MIT-BIH record.")
        print("Reason:", e)

In [ ]:
if "record" in globals():
    ecg = record.p_signal[:, 0]

    plt.figure(figsize=(12, 4))
    plt.plot(ecg[:5000])
    plt.title("ECG Signal — First Channel")
    plt.xlabel("Sample")
    plt.ylabel("Amplitude")
    plt.show()

In [ ]:
def bandpass_filter(ecg_signal, fs, low=0.5, high=40.0, order=4):
    nyquist = fs / 2
    high = min(high, nyquist * 0.95)

    b, a = signal.butter(
        order,
        [low / nyquist, high / nyquist],
        btype="band"
    )
    return signal.filtfilt(b, a, ecg_signal)

if "record" in globals():
    filtered_ecg = bandpass_filter(ecg, record.fs)

    plt.figure(figsize=(12, 4))
    plt.plot(ecg[:3000], label="Original")
    plt.plot(filtered_ecg[:3000], label="Filtered")
    plt.title("ECG Noise Removal")
    plt.xlabel("Sample")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.show()

In [ ]:
def create_windows(signal_data, window_size, step):
    windows = []

    for start in range(0, len(signal_data) - window_size + 1, step):
        window = signal_data[start:start + window_size]
        windows.append(window)

    return np.array(windows)

if "record" in globals():
    window_size = int(record.fs * 5)  # 5-second windows
    step = int(record.fs * 2.5)      # 50% overlap

    ecg_windows = create_windows(filtered_ecg, window_size, step)

    print("Windowed ECG shape:", ecg_windows.shape)
    if len(ecg_windows) > 0:
        plt.figure(figsize=(12, 4))
        plt.plot(ecg_windows[0])
        plt.title("First ECG Window")
        plt.xlabel("Sample")
        plt.ylabel("Amplitude")
        plt.show()

In [ ]:
if "wfdb" in globals():
    try:
        annotation = wfdb.rdann(RECORD_PATH, "atr")
        print("Number of annotations:", len(annotation.sample))
        print("First annotation samples:", annotation.sample[:20])
        print("First annotation symbols:", annotation.symbol[:20])
    except Exception as e:
        print("Annotations not loaded. Ensure the corresponding .atr file exists.")
        print("Reason:", e)